In [ ]:
import time
import numpy as np
import pandas as pd
from pyecharts.charts import Bar3D

# 1. 读取数据

In [ ]:
# 1. 定义变量, 记录表名.
sheet_names = ['2015', '2016', '2017', '2018', '会员等级']

# 2. 读取上述的excel表,
# 参1: 文件名, 参2: sheet名
sheet_dict = pd.read_excel('./data/sales.xlsx', sheet_name=sheet_names)
sheet_dict

In [ ]:
# 3. 从字典中 获取 2015表 的数据.
sheet_dict['2015']

In [ ]:
# 4. 查看数据的分布情况
# 4.1 遍历获取每个表名
for sheet_name in sheet_names:
    # 4.2 打印当前表的统计信息
    print(sheet_dict[sheet_name].describe())
    # 4.3 打印当前表的数据结构
    print(sheet_dict[sheet_name].info())
    print('-' * 50)


# 2.数据的预处理

In [ ]:
# 获取前4张表的所有 数据
for i in sheet_names[:-1]:
    # 处理缺失值
    sheet_dict[i] = sheet_dict[i].dropna()
    # 删除订单金额小于1的行
    sheet_dict[i] = sheet_dict[i][sheet_dict[i]['订单金额'] > 1]
    # 新增一列, 用于表示: 固定时间(统计时间)
    sheet_dict[i]['max_year_date'] = sheet_dict[i]['提交日期'].max()

In [ ]:
for i in sheet_names:
    print(sheet_dict[i].info())        # 表的详细信息
    print(sheet_dict[i].describe())    # 表的统计信息

In [12]:
data_merge = pd.concat(list(sheet_dict.values())[:-1], ignore_index=True)
data_merge['year'] = data_merge['max_year_date'].dt.year
data_merge['date_interval'] = data_merge['max_year_date'] - data_merge['提交日期']
data_merge['date_interval'] = data_merge['date_interval'].dt.days
data_merge


,会员ID,订单号,提交日期,订单金额,max_year_date,year,date_interval
0,15278002468,3000304681,2015-01-01,499.0,2015-12-31,2015,364
1,39236378972,3000305791,2015-01-01,2588.0,2015-12-31,2015,364
2,38722039578,3000641787,2015-01-01,498.0,2015-12-31,2015,364
3,11049640063,3000798913,2015-01-01,1572.0,2015-12-31,2015,364
4,35038752292,3000821546,2015-01-01,10.1,2015-12-31,2015,364
...,...,...,...,...,...,...,...
202822,39229485704,4354225182,2018-12-31,249.0,2018-12-31,2018,0
202823,39229021075,4354225188,2018-12-31,89.0,2018-12-31,2018,0
202824,39288976750,4354230034,2018-12-31,48.5,2018-12-31,2018,0
202825,26772630,4354230163,2018-12-31,3196.0,2018-12-31,2018,0


# 3. 数据统计分析

In [20]:
rfm_gb = data_merge.groupby(['year', '会员ID'], as_index=False).agg({
    'date_interval': 'min',
    '订单号': 'count',
    '订单金额': 'sum'
})
rfm_gb.columns = ['year', '会员ID', 'recency', 'frequency', 'monetary']
rfm_gb

,year,会员ID,recency,frequency,monetary
0,2015,267,197,2,105.0
1,2015,282,251,1,29.7
2,2015,283,340,1,5398.0
3,2015,343,300,1,118.0
4,2015,525,37,3,213.0
...,...,...,...,...,...
148586,2018,39538034299,272,1,49.0
148587,2018,39538034662,189,1,3558.0
148588,2018,39538035729,179,1,3699.0
148589,2018,39545237824,275,1,49.0


In [32]:
print(rfm_gb.iloc[:, 2:].describe().T)
r_bins = [-1, 79, 255, 365]
f_bins = [0, 2, 5, 130]
m_bins = [0, 69, 1199, 206252]

              count         mean          std  min   25%    50%     75%  \
recency    148591.0   165.524043   101.988472  0.0  79.0  156.0   255.0   
frequency  148591.0     1.365002     2.626953  1.0   1.0    1.0     1.0   
monetary   148591.0  1323.741329  3753.906883  1.5  69.0  189.0  1199.0   

                max  
recency       365.0  
frequency     130.0  
monetary   206251.8  


In [39]:
pd.cut(rfm_gb['recency'], r_bins, labels=['r1', 'r2', 'r3']).unique()
# pd.cut(rfm_gb['recency'], bins=3, labels=['r1', 'r2', 'r3']).unique()

['r2', 'r3', 'r1']
Categories (3, object): ['r1' < 'r2' < 'r3']